In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
import glob
import geopandas as gpd

# =============================================
# 1. AGREGAR CASOS DO SINAN POR MUNICÍPIO × MÊS
# =============================================

# Carregar todos os anos
sinan_files = glob.glob('../data/raw/sinan/leptospirose_*.parquet')
if sinan_files:
    df_sinan = pd.concat([pd.read_parquet(f) for f in sinan_files])

    # Filtrar apenas casos confirmados (CLASSI_FIN == 1)
    df_conf = df_sinan[df_sinan['CLASSI_FIN'] == 1].copy()

    # Criar coluna de data
    df_conf['dt_notific'] = pd.to_datetime(df_conf['DT_NOTIFIC'], errors='coerce')
    df_conf['ano_mes'] = df_conf['dt_notific'].dt.to_period('M')
    df_conf['code_muni'] = df_conf['ID_MUNICIP'].astype(str).str[:6]

    # Agregar: casos por município × mês
    casos_muni_mes = df_conf.groupby(['code_muni', 'ano_mes']).size().reset_index(name='casos_leptospirose')
    print(f"Leptospirose: {casos_muni_mes.shape[0]} registros município×mês")
else:
    print("Arquivos de leptospirose não encontrados.")

# Repetir para leishmaniose
sinan_leiv = glob.glob('../data/raw/sinan/leish_visceral_*.parquet')
if sinan_leiv:
    df_leiv = pd.concat([pd.read_parquet(f) for f in sinan_leiv])
    df_leiv_conf = df_leiv[df_leiv['CLASSI_FIN'] == 1].copy()
    df_leiv_conf['dt_notific'] = pd.to_datetime(df_leiv_conf['DT_NOTIFIC'], errors='coerce')
    df_leiv_conf['ano_mes'] = df_leiv_conf['dt_notific'].dt.to_period('M')
    df_leiv_conf['code_muni'] = df_leiv_conf['ID_MUNICIP'].astype(str).str[:6]
    casos_leiv = df_leiv_conf.groupby(['code_muni', 'ano_mes']).size().reset_index(name='casos_leishmaniose')
    print(f"Leishmaniose: {casos_leiv.shape[0]} registros município×mês")
else:
    print("Arquivos de leishmaniose não encontrados.")

